# Fixed UE versus fixed PRB allocation

This notebook tests an important distinction in the simulator:

- **Fixed UE** means its position and serving gNB do not change.
- It does **not** necessarily mean its scheduler allocation remains constant.

We disable handovers, keep six eMBB UEs fixed on the center cell, and record both per-UE useful PRBs and physical gNB utilization over time.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'global_ppo_3gnb_env.py').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from global_ppo_3gnb_env import GlobalPPO3GNBEnv

## Create the fixed-UE scenario

All six eMBB UEs have zero speed and start attached to center gNB1. A `+6 dB` A3 offset is applied in every direction and safe admission receives zero release bias, so no handover can execute during this test.

In [ ]:
env = GlobalPPO3GNBEnv(
    seed=7,
    scenario_mode='curriculum',
    training_scenarios='fixed_center_embb_left_right',
    scenario_selection='cycle',
    upper_window_seconds=1.0,
    local_steps_per_global=10,
    radio_substeps=100,
    radio_tick_seconds=0.001,
    pf_averaging_window_s=0.25,
    terminal_reward_only=False,
    safe_admission_enabled=True,
)

observation, reset_info = env.reset(seed=7)
ues = sorted(env.base_env.get_all_ues(), key=lambda ue: int(ue.id))
initial_state = {
    int(ue.id): (float(ue.x), float(ue.y), int(ue.serving_gnb))
    for ue in ues
}

retain_offsets = np.full((3, 2, 3), 6.0, dtype=float)
zero_bias = np.zeros((3, 2, 3), dtype=np.float32)
env._apply_slice_offsets(retain_offsets)
env.base_env.begin_safe_admission_window(zero_bias, env.slice_types)

print('UE IDs:', [int(ue.id) for ue in ues])
print('Initial serving gNBs:', [int(ue.serving_gnb) for ue in ues])
print('Physical PRBs per gNB:', [int(gnb.n_prbs) for gnb in env.base_env.gnbs])

## Advance the scheduler without changing mobility

Each row below is one local simulator step. `useful_prbs` is the allocation that actually carried useful data during the most recent radio service interval.

In [ ]:
records = []
n_samples = 40

for sample in range(n_samples):
    env.base_env.step(0)
    row = {'sample': sample}
    for ue in ues:
        row[f'UE{int(ue.id)} useful PRBs'] = int(ue.useful_prbs)
        row[f'UE{int(ue.id)} allocated PRBs'] = int(ue.prbs)
    for gnb_id in range(3):
        row[f'gNB{gnb_id} eMBB utilization'] = float(
            env.base_env.estimate_slice_load(gnb_id, 'eMBB')
        )
    records.append(row)

trace = pd.DataFrame(records)
trace.head(10)

## Verify that mobility really stayed fixed

In [ ]:
final_state = {
    int(ue.id): (float(ue.x), float(ue.y), int(ue.serving_gnb))
    for ue in ues
}

assert final_state == initial_state
assert len(env.base_env.get_handover_events()) == 0
print('PASS: every UE kept the same position and serving gNB; handovers = 0.')

## Did each UE keep the same PRB allocation?

A non-zero standard deviation means that UE's useful allocation changed even though the UE itself did not move.

In [ ]:
useful_columns = [f'UE{int(ue.id)} useful PRBs' for ue in ues]
prb_summary = pd.DataFrame({
    'minimum': trace[useful_columns].min(),
    'maximum': trace[useful_columns].max(),
    'mean': trace[useful_columns].mean(),
    'standard deviation': trace[useful_columns].std(ddof=0),
})

varying_ues = prb_summary.index[prb_summary['standard deviation'] > 0].tolist()
assert varying_ues, 'This deterministic run unexpectedly produced no scheduler variation.'
display(prb_summary.style.format('{:.2f}'))
print('UE allocations that changed:', varying_ues)

In [ ]:
ax = trace.plot(
    x='sample', y=useful_columns, figsize=(12, 5), marker='o', markersize=3
)
ax.set(title='Useful PRBs vary although UE positions and attachments are fixed',
       xlabel='local scheduler sample', ylabel='useful PRBs')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Aggregate allocated utilization

The upper agent observes slice-level utilization:

$$U_{i,s}=\frac{\sum_{u\in(i,s)}\mathrm{usefulPRBs}_u}{100}.$$

Individual allocations can move between UEs while aggregate utilization remains stable. In this scenario, gNB1 is continuously saturated, so its aggregate eMBB utilization stays near `1.0`.

In [ ]:
util_columns = [f'gNB{i} eMBB utilization' for i in range(3)]
util_summary = pd.DataFrame({
    'minimum': trace[util_columns].min(),
    'maximum': trace[util_columns].max(),
    'mean': trace[util_columns].mean(),
    'standard deviation': trace[util_columns].std(ddof=0),
})
display(util_summary.style.format('{:.3f}'))

ax = trace.plot(x='sample', y=util_columns, figsize=(12, 4), ylim=(-0.03, 1.05))
ax.set(title='Physical allocated utilization per gNB',
       xlabel='local scheduler sample', ylabel='useful allocated PRBs / 100')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Load of each gNB and slice percentages

Two percentages are intentionally reported:

- **Physical utilization (%)** = slice useful PRBs / physical gNB PRBs.
- **Share inside gNB (%)** = slice useful PRBs / all useful PRBs currently allocated by that gNB.

The first measures physical load. The second shows how the currently allocated load is divided among eMBB, URLLC, and mMTC.

In [ ]:
slice_types = list(env.slice_types)
load_rows = []

for gnb_id in range(env.n_gnbs):
    gnb = env.base_env._get_gnb_by_id(gnb_id)
    physical_prbs = int(gnb.n_prbs)
    slice_used = {
        slice_type: int(env.base_env.get_slice_used_prbs(gnb_id, slice_type))
        for slice_type in slice_types
    }
    total_used = sum(slice_used.values())

    for slice_type in slice_types:
        used = slice_used[slice_type]
        load_rows.append({
            'gNB': f'gNB{gnb_id}',
            'slice': slice_type,
            'useful allocated PRBs': used,
            'physical gNB PRBs': physical_prbs,
            'physical utilization (%)': 100.0 * used / physical_prbs,
            'share inside gNB (%)': (
                100.0 * used / total_used if total_used > 0 else 0.0
            ),
            'total gNB utilization (%)': 100.0 * total_used / physical_prbs,
        })

gnb_slice_load = pd.DataFrame(load_rows)
display(gnb_slice_load.style.format({
    'physical utilization (%)': '{:.1f}',
    'share inside gNB (%)': '{:.1f}',
    'total gNB utilization (%)': '{:.1f}',
}))

# Physical conservation: all slice allocations together cannot exceed 100%.
assert np.all(
    gnb_slice_load.groupby('gNB')['physical utilization (%)'].sum() <= 100.0 + 1e-9
)
gnb_slice_load

In [ ]:
physical_by_slice = gnb_slice_load.pivot(
    index='gNB', columns='slice', values='physical utilization (%)'
).reindex(columns=slice_types, fill_value=0.0)

ax = physical_by_slice.plot(
    kind='bar', stacked=True, figsize=(10, 5),
    color={'eMBB': '#f4a261', 'URLLC': '#e63946', 'mMTC': '#6a4c93'},
)
ax.axhline(100.0, color='black', linestyle='--', linewidth=1, label='physical capacity')
ax.set(title='Physical gNB utilization divided by slice',
       xlabel='gNB', ylabel='allocated utilization (%)', ylim=(0, 108))
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.25)
ax.legend(title='slice')
plt.tight_layout()
plt.show()

## Upper-window averaging

For a stable upper-agent state, average allocated utilization over several scheduler samples rather than relying on one instantaneous allocation.

In [ ]:
window = 10
rolling = trace[util_columns].rolling(window=window).mean()
ax = rolling.plot(figsize=(12, 4), ylim=(-0.03, 1.05))
ax.set(title=f'{window}-sample moving average of allocated utilization',
       xlabel='local scheduler sample', ylabel='mean utilization')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print('Conclusion: fixed mobility does not guarantee fixed per-UE PRBs.')
print('The upper controller should use window-averaged allocated utilization.')

In [ ]:
env.close()

# Scenario 2: different slices distributed across all gNBs

This second fixed-mobility experiment uses `mixed_overlap_with_fixed_slice_loads`:

- gNB0 carries fixed-core **mMTC** traffic.
- gNB1 carries mixed **eMBB** and **URLLC** traffic.
- gNB2 carries fixed-core **URLLC** traffic.

Again, handovers are disabled. The table uses the mean allocated utilization across 20 scheduler samples.

In [ ]:
mixed_env = GlobalPPO3GNBEnv(
    seed=11,
    scenario_mode='curriculum',
    training_scenarios='mixed_overlap_with_fixed_slice_loads',
    scenario_selection='cycle',
    upper_window_seconds=1.0,
    local_steps_per_global=10,
    radio_substeps=100,
    radio_tick_seconds=0.001,
    pf_averaging_window_s=0.25,
    terminal_reward_only=False,
    safe_admission_enabled=True,
)
mixed_env.reset(seed=11)

mixed_env._apply_slice_offsets(np.full((3, 2, 3), 6.0, dtype=float))
mixed_env.base_env.begin_safe_admission_window(
    np.zeros((3, 2, 3), dtype=np.float32), mixed_env.slice_types
)
mixed_ues = sorted(mixed_env.base_env.get_all_ues(), key=lambda ue: int(ue.id))
mixed_initial_state = {
    int(ue.id): (float(ue.x), float(ue.y), int(ue.serving_gnb))
    for ue in mixed_ues
}

ue_count_table = pd.DataFrame(
    [[mixed_env.base_env.get_slice_ue_count(g, s) for s in mixed_env.slice_types]
     for g in range(3)],
    index=[f'gNB{g}' for g in range(3)],
    columns=mixed_env.slice_types,
)
ue_count_table.index.name = 'serving gNB'
display(ue_count_table)
ue_count_table

In [ ]:
mixed_samples = []
for sample in range(20):
    mixed_env.base_env.step(0)
    mixed_samples.append([
        [mixed_env.base_env.estimate_slice_load(g, s) for s in mixed_env.slice_types]
        for g in range(3)
    ])

mixed_mean_load = np.mean(np.asarray(mixed_samples, dtype=float), axis=0)
mixed_final_state = {
    int(ue.id): (float(ue.x), float(ue.y), int(ue.serving_gnb))
    for ue in mixed_ues
}
assert mixed_final_state == mixed_initial_state
assert len(mixed_env.base_env.get_handover_events()) == 0
assert np.all(mixed_mean_load.sum(axis=1) <= 1.0 + 1e-9)
print('PASS: UE positions and serving cells remained fixed; handovers = 0.')

In [ ]:
mixed_rows = []
for gnb_id in range(3):
    total_utilization = float(mixed_mean_load[gnb_id].sum())
    for slice_idx, slice_type in enumerate(mixed_env.slice_types):
        utilization = float(mixed_mean_load[gnb_id, slice_idx])
        mixed_rows.append({
            'gNB': f'gNB{gnb_id}',
            'slice': slice_type,
            'UE count': int(ue_count_table.loc[f'gNB{gnb_id}', slice_type]),
            'mean useful PRBs': 100.0 * utilization,
            'physical utilization (%)': 100.0 * utilization,
            'share inside gNB (%)': (
                100.0 * utilization / total_utilization
                if total_utilization > 0 else 0.0
            ),
            'total gNB utilization (%)': 100.0 * total_utilization,
        })

mixed_gnb_slice_load = pd.DataFrame(mixed_rows)
display(mixed_gnb_slice_load.style.format({
    'mean useful PRBs': '{:.1f}',
    'physical utilization (%)': '{:.1f}',
    'share inside gNB (%)': '{:.1f}',
    'total gNB utilization (%)': '{:.1f}',
}))
mixed_gnb_slice_load

In [ ]:
mixed_plot = pd.DataFrame(
    100.0 * mixed_mean_load,
    index=[f'gNB{i}' for i in range(3)],
    columns=mixed_env.slice_types,
)
ax = mixed_plot.plot(
    kind='bar', stacked=True, figsize=(10, 5),
    color={'eMBB': '#f4a261', 'URLLC': '#e63946', 'mMTC': '#6a4c93'},
)
ax.axhline(100.0, color='black', linestyle='--', linewidth=1, label='physical capacity')
ax.set(title='Scenario 2: mean physical utilization by gNB and slice',
       xlabel='gNB', ylabel='allocated utilization (%)', ylim=(0, 108))
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.25)
ax.legend(title='slice')
plt.tight_layout()
plt.show()

print('Mean utilization matrix [gNB, slice]:')
display(mixed_plot.style.format('{:.1f}'))
mixed_env.close()